# Notebook 6: 05_sql_analysis
SQL-focused analysis using sqlite3.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
DB_PATH = DATA_DIR / 'fireworks_monitoring_cleaned.db'
if not DB_PATH.exists():
    DB_PATH = DATA_DIR / 'fireworks_monitoring.db'
conn = sqlite3.connect(DB_PATH)
print('Using DB:', DB_PATH)


In [ ]:
def run_sql(sql):
    return pd.read_sql_query(sql, conn)


In [ ]:
# SQL 1: by scene sample count and FP rate
sql1 = """
SELECT
    scene_type,
    COUNT(*) AS total_samples,
    SUM(CASE WHEN error_type = 'FP' THEN 1 ELSE 0 END) AS fp_count,
    ROUND(1.0 * SUM(CASE WHEN error_type = 'FP' THEN 1 ELSE 0 END) / COUNT(*), 4) AS fp_rate
FROM audio_clip_event
GROUP BY scene_type
ORDER BY fp_rate DESC;
"""
run_sql(sql1)


In [ ]:
# SQL 2: FP source top categories
sql2 = """
SELECT
    true_class,
    COUNT(*) AS fp_count
FROM audio_clip_event
WHERE error_type = 'FP'
GROUP BY true_class
ORDER BY fp_count DESC;
"""
run_sql(sql2)


In [ ]:
# SQL 3: alert count by time bucket
sql3 = """
SELECT
    time_bucket,
    COUNT(*) AS alert_count
FROM alert_record
GROUP BY time_bucket
ORDER BY alert_count DESC;
"""
run_sql(sql3)


In [ ]:
# SQL 4: threshold version performance
sql4 = """
SELECT
    threshold_version,
    ROUND(AVG(score_at_alert), 4) AS avg_alert_score,
    COUNT(*) AS alert_volume,
    SUM(CASE WHEN operator_review_result = 'false_alarm' THEN 1 ELSE 0 END) AS false_alarm_count
FROM alert_record
GROUP BY threshold_version;
"""
run_sql(sql4)


In [ ]:
# SQL 5: device FP ranking
sql5 = """
SELECT
    device_id,
    COUNT(*) AS total_samples,
    SUM(CASE WHEN error_type = 'FP' THEN 1 ELSE 0 END) AS fp_count
FROM audio_clip_event
GROUP BY device_id
ORDER BY fp_count DESC
LIMIT 10;
"""
run_sql(sql5)


In [ ]:
# SQL 6: join query for scene + time bucket FP hotspots
sql6 = """
SELECT
    a.scene_type,
    a.time_bucket,
    COUNT(*) AS total_fp
FROM audio_clip_event a
JOIN device_info d
ON a.device_id = d.device_id
WHERE a.error_type = 'FP'
GROUP BY a.scene_type, a.time_bucket
ORDER BY total_fp DESC;
"""
run_sql(sql6)


In [ ]:
conn.close()
